# Efficient Multi-Scale Visual Saliency Prediction

This notebook implements a PyTorch project for visual saliency prediction on the SALICON dataset.

The project compares:

- **MeanMap**: non-learned centre-prior baseline
- **Light-S**: MobileNetV2 with a single-scale decoder
- **Light-M**: MobileNetV2 with a compact multi-scale decoder
- **Heavy-M**: ResNet-18 with the same multi-scale decoder

The main objective is to study the trade-off between prediction quality and computational efficiency, including robustness on off-centre scenes.

All experiments are executed in Google Colab, while datasets, checkpoints, logs, and generated results are stored persistently in Google Drive.

In [47]:
# ---------------------------------------------------------
# Mount Google Drive and verify the real mount
# ---------------------------------------------------------

from google.colab import drive
from pathlib import Path
import os
import subprocess

MOUNT_POINT = Path("/content/drive")

# Remove any previous or disconnected mount.
try:
    drive.flush_and_unmount()
except Exception:
    pass

# Mount again and force a fresh connection.
drive.mount(str(MOUNT_POINT), force_remount=True)

# Verify that MyDrive is available.
MY_DRIVE = MOUNT_POINT / "MyDrive"

if not MY_DRIVE.is_dir():
    raise RuntimeError(
        "Google Drive mount failed: /content/drive/MyDrive does not exist."
    )

# Check the filesystem containing MyDrive.
mount_result = subprocess.run(
    ["findmnt", "-T", str(MY_DRIVE)],
    capture_output=True,
    text=True,
)

print("MyDrive exists:", MY_DRIVE.is_dir())
print("Mount information:")
print(mount_result.stdout or mount_result.stderr)

print("\nTop-level Drive contents:")
for item in sorted(MY_DRIVE.iterdir())[:20]:
    print(" -", item.name)

Mounted at /content/drive
MyDrive exists: True
Mount information:
TARGET         SOURCE FSTYPE     OPTIONS
/content/drive drive  fuse.drive rw,nosuid,nodev,relatime,user_id=0,group_id=0


Top-level Drive contents:
 - COSE CRI.gdoc
 - Cartella senza titolo
 - Colab Notebooks
 - Computer Sceince
 - Copy of ai_companies_with_linkedin.gsheet
 - Documento senza titolo.gdoc
 - ERASMUS-SMP-GRANT-CALCULATOR new (1).xlsx
 - Efficient_Saliency_Project_Baseline_Report (1).gdoc
 - Efficient_Saliency_Project_Baseline_Report.gdoc
 - Efficient_Saliency_Project_Foundation (1).gdoc
 - Efficient_Saliency_Project_Foundation.gdoc
 - Expense_Traker.gsheet
 - FOTO CRI
 - Fanta.gsheet
 - File_000
 - Foglio Spese.gsheet
 - Foto Laurea Sara 
 - GxxHW2form.gdoc
 - Imec.gdoc
 - JUnit.pdf


##Project directory structure

The project uses a simple Drive-based structure.

- `data/` stores the SALICON dataset
- `checkpoints/` stores trained model weights
- `outputs/` stores figures, metrics, and predictions
- `logs/` stores experiment logs
- `splits/` stores deterministic train, validation, and test IDs
- `notebooks/` stores notebook backups

The dataset and checkpoints are not stored inside the temporary `/content` directory.

In [48]:
# ---------------------------------------------------------
# Canonical persistent Google Drive paths
# ---------------------------------------------------------

from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/saliency_project")

# Raw data
DATA_ROOT = DRIVE_ROOT / "data" / "SALICON"

# Project outputs
CHECKPOINT_ROOT = DRIVE_ROOT / "checkpoints"
OUTPUT_ROOT = DRIVE_ROOT / "outputs"
FIGURE_ROOT = OUTPUT_ROOT / "figures"
METRICS_ROOT = OUTPUT_ROOT / "metrics"
PREDICTION_ROOT = OUTPUT_ROOT / "predictions"

# Project metadata and code
LOG_ROOT = DRIVE_ROOT / "logs"
SPLIT_ROOT = DRIVE_ROOT / "splits"
NOTEBOOK_ROOT = DRIVE_ROOT / "notebooks"
BACKUP_ROOT = DRIVE_ROOT / "backups"

PROJECT_PATHS = [
    DATA_ROOT,

    CHECKPOINT_ROOT / "light_single",
    CHECKPOINT_ROOT / "light_multi",
    CHECKPOINT_ROOT / "heavy_multi",

    FIGURE_ROOT,
    METRICS_ROOT,
    PREDICTION_ROOT,

    LOG_ROOT,
    SPLIT_ROOT,
    NOTEBOOK_ROOT,
    BACKUP_ROOT,
]

for path in PROJECT_PATHS:
    path.mkdir(parents=True, exist_ok=True)

assert str(DRIVE_ROOT).startswith("/content/drive/MyDrive/"), (
    f"DRIVE_ROOT is not inside mounted Google Drive: {DRIVE_ROOT}"
)

print("Project root:       ", DRIVE_ROOT)
print("Dataset root:       ", DATA_ROOT)
print("Splits root:        ", SPLIT_ROOT)
print("Logs root:          ", LOG_ROOT)
print("Figures root:       ", FIGURE_ROOT)
print("Checkpoints root:   ", CHECKPOINT_ROOT)

Project root:        /content/drive/MyDrive/saliency_project
Dataset root:        /content/drive/MyDrive/saliency_project/data/SALICON
Splits root:         /content/drive/MyDrive/saliency_project/splits
Logs root:           /content/drive/MyDrive/saliency_project/logs
Figures root:        /content/drive/MyDrive/saliency_project/outputs/figures
Checkpoints root:    /content/drive/MyDrive/saliency_project/checkpoints


In [49]:
# ---------------------------------------------------------
# Persistent Drive write test
# ---------------------------------------------------------

from datetime import datetime, timezone
import os

timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

TEST_PATH = DRIVE_ROOT / f"DRIVE_TEST_{timestamp}.txt"

TEST_PATH.write_text(
    "This file was written through the mounted Google Drive.\n",
    encoding="utf-8",
)

os.sync()

print("Created file:")
print(TEST_PATH)
print("Exists:", TEST_PATH.is_file())
print()
print("Search Google Drive in your browser for this exact filename:")
print(TEST_PATH.name)

Created file:
/content/drive/MyDrive/saliency_project/DRIVE_TEST_20260712_143643.txt
Exists: True

Search Google Drive in your browser for this exact filename:
DRIVE_TEST_20260712_143643.txt


In [50]:
import os
import platform
import subprocess
import torch
import torchvision

print("Python version:", platform.python_version())
print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB",
    )
else:
    print("GPU not detected. Enable a GPU runtime before training.")

print("\nCPU information:")
cpu_info = subprocess.run(
    ["bash", "-lc", "lscpu | grep 'Model name' | head -1"],
    capture_output=True,
    text=True,
)
print(cpu_info.stdout.strip())

print("\nStorage information:")
os.system("df -h /content /content/drive | tail -n +2")

Python version: 3.12.13
PyTorch version: 2.11.0+cu128
Torchvision version: 0.26.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8
GPU memory: 14.56 GB

CPU information:
Model name:                              Intel(R) Xeon(R) CPU @ 2.00GHz

Storage information:


0

In [51]:
import random
import numpy as np

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Random seed:", SEED)
print("Selected device:", DEVICE)

Random seed: 42
Selected device: cuda


# SALICON Dataset Setup and Audit

## Objective

Before implementing the neural network, we verify that the SALICON dataset is
available, correctly structured, and internally consistent.

This stage checks:

- where the RGB images and saliency maps are stored;
- how filenames identify corresponding image-map pairs;
- whether any files are missing or duplicated;
- image and map dimensions, channels, data types, and value ranges;
- whether the saliency maps are continuous distributions;
- whether images and maps remain spatially aligned.

## Why this is necessary

An incorrect image-map pairing would cause the network to learn meaningless
relationships. Dataset problems must therefore be detected before creating
splits, DataLoaders, losses, or models.

## Required outputs

- `logs/data_audit.json`
- `outputs/figures/data_contact_sheet.png`
- a verified list of valid image-map pairs

## Stop condition

We do not continue to model implementation until the dataset pairing and
visual alignment are confirmed.

In [9]:
from google.colab import userdata

# Read the token from Colab Secrets
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
!pip install -q --upgrade kaggle
import kaggle

In [10]:
# ---------------------------------------------------------
# Verify Kaggle authentication and dataset access
# ---------------------------------------------------------

import subprocess

dataset_id = "roshan401/salicon"

result = subprocess.run(
    [
        "kaggle",
        "datasets",
        "files",
        "-d",
        dataset_id,
    ],
    check=False,
    text=True,
    capture_output=True,
)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(
        "Kaggle authentication or dataset access failed. "
        "Check the token and ensure notebook access is enabled "
        "in the Colab Secrets panel."
    )

print(result.stdout)

Next Page Token = CfDJ8HMxIwPZ_4ZImnEqAVaKkSGXHIVD31FDkV_-hp1G_o6TFqcY-WV3TYr_kkZp8x1qfHci9ui6ObMEPL0bjrHckyVEqeaI5Co7rQHB6s-qiEH6L3Hhd7JcaH96bkoPw8zWj8Wgld7GX8sLwphux7PwHXazxJoLTFszd9mUPLfOYnlLBtxsPA
name                                           size  creationDate                
---------------------------------------------  ----  --------------------------  
fixations/test/COCO_test2014_000000000001.mat   256  2024-04-12 06:40:32.949000  
fixations/test/COCO_test2014_000000000063.mat   257  2024-04-12 06:40:38.773000  
fixations/test/COCO_test2014_000000000188.mat   258  2024-04-12 06:40:38.422000  
fixations/test/COCO_test2014_000000000219.mat   258  2024-04-12 06:40:37.795000  
fixations/test/COCO_test2014_000000000318.mat   258  2024-04-12 06:40:36.530000  
fixations/test/COCO_test2014_000000000535.mat   258  2024-04-12 06:40:36.751000  
fixations/test/COCO_test2014_000000000549.mat   258  2024-04-12 06:40:36.966000  
fixations/test/COCO_test2014_000000000627.mat   258  2024-04-

In [11]:
# ---------------------------------------------------------
# Prepare persistent SALICON destination
# ---------------------------------------------------------

from pathlib import Path

DATA_ROOT.mkdir(parents=True, exist_ok=True)

existing_files = [
    path for path in DATA_ROOT.rglob("*")
    if path.is_file()
]

print(f"Destination: {DATA_ROOT}")
print(f"Existing files: {len(existing_files)}")

if existing_files:
    raise RuntimeError(
        "DATA_ROOT already contains files. "
        "Stop here and inspect them before downloading again."
    )

print("Destination is empty and ready.")

Destination: /content/drive/MyDrive/saliency_project/data/SALICON
Existing files: 0
Destination is empty and ready.


In [12]:
# ---------------------------------------------------------
# Download and extract SALICON into Google Drive
# ---------------------------------------------------------

import subprocess

dataset_id = "roshan401/salicon"

print(f"Dataset: {dataset_id}")
print(f"Destination: {DATA_ROOT}")
print("Download starting. Do not disconnect the runtime.")

subprocess.run(
    [
        "kaggle",
        "datasets",
        "download",
        "-d",
        dataset_id,
        "-p",
        str(DATA_ROOT),
        "--unzip",
    ],
    check=True,
)

print("SALICON download and extraction completed.")

Dataset: roshan401/salicon
Destination: /content/drive/MyDrive/saliency_project/data/SALICON
Download starting. Do not disconnect the runtime.
SALICON download and extraction completed.


In [52]:
# ---------------------------------------------------------
# Verify extracted dataset
# ---------------------------------------------------------

from collections import Counter

all_files = sorted(
    path for path in DATA_ROOT.rglob("*")
    if path.is_file()
)

if not all_files:
    raise RuntimeError(
        f"No files were found under {DATA_ROOT}. "
        "The download may not have completed."
    )

total_bytes = sum(path.stat().st_size for path in all_files)

extension_counts = Counter(
    path.suffix.lower() if path.suffix else "<no extension>"
    for path in all_files
)

print(f"Total files: {len(all_files):,}")
print(f"Total size: {total_bytes / (1024 ** 3):.2f} GB")

print("\nFile extensions:")
for extension, count in extension_counts.most_common():
    print(f"  {extension}: {count:,}")

print("\nTop-level contents:")
for item in sorted(DATA_ROOT.iterdir()):
    kind = "DIR " if item.is_dir() else "FILE"
    print(f"  [{kind}] {item.name}")

Total files: 55,000
Total size: 3.97 GB

File extensions:
  .mat: 20,000
  .jpg: 20,000
  .png: 15,000

Top-level contents:
  [DIR ] fixations
  [DIR ] images
  [DIR ] maps


# 5. SALICON Dataset Structure Audit

## Objective

Inspect the downloaded SALICON package and identify the exact locations and
filename conventions of RGB images, continuous saliency maps, and fixation
annotations.

For the initial training pipeline, RGB `.jpg` images will be paired with
continuous `.png` saliency maps. The `.mat` fixation files will remain
untouched and are not required for the primary KLD, CC, and SIM metrics.

## Why this audit is necessary

The neural network requires every image to be paired with its corresponding
ground-truth saliency map. Pairing files incorrectly would invalidate the
entire experiment.

We therefore inspect paths and filenames before creating dataset splits or
implementing the PyTorch Dataset.

In [53]:
# ---------------------------------------------------------
# Collect SALICON files by type
# ---------------------------------------------------------

jpg_files = sorted(DATA_ROOT.rglob("*.jpg"))
png_files = sorted(DATA_ROOT.rglob("*.png"))
mat_files = sorted(DATA_ROOT.rglob("*.mat"))

print(f"RGB images (.jpg):       {len(jpg_files):,}")
print(f"Saliency maps (.png):    {len(png_files):,}")
print(f"Fixation files (.mat):   {len(mat_files):,}")

if not jpg_files:
    raise RuntimeError("No JPG images were found.")

if not png_files:
    raise RuntimeError("No PNG saliency maps were found.")

print("\nExample JPG paths:")
for path in jpg_files[:10]:
    print(" ", path.relative_to(DATA_ROOT))

print("\nExample PNG paths:")
for path in png_files[:10]:
    print(" ", path.relative_to(DATA_ROOT))

print("\nExample MAT paths:")
for path in mat_files[:10]:
    print(" ", path.relative_to(DATA_ROOT))

RGB images (.jpg):       20,000
Saliency maps (.png):    15,000
Fixation files (.mat):   20,000

Example JPG paths:
  images/images/test/COCO_test2014_000000000001.jpg
  images/images/test/COCO_test2014_000000000063.jpg
  images/images/test/COCO_test2014_000000000188.jpg
  images/images/test/COCO_test2014_000000000219.jpg
  images/images/test/COCO_test2014_000000000318.jpg
  images/images/test/COCO_test2014_000000000535.jpg
  images/images/test/COCO_test2014_000000000549.jpg
  images/images/test/COCO_test2014_000000000627.jpg
  images/images/test/COCO_test2014_000000000694.jpg
  images/images/test/COCO_test2014_000000001043.jpg

Example PNG paths:
  maps/train/COCO_train2014_000000000009.png
  maps/train/COCO_train2014_000000000089.png
  maps/train/COCO_train2014_000000000110.png
  maps/train/COCO_train2014_000000000307.png
  maps/train/COCO_train2014_000000000321.png
  maps/train/COCO_train2014_000000000332.png
  maps/train/COCO_train2014_000000000349.png
  maps/train/COCO_train2014_0

In [54]:
# ---------------------------------------------------------
# Summarize dataset folders and file extensions
# ---------------------------------------------------------

from collections import Counter

all_directories = [DATA_ROOT] + sorted(
    path for path in DATA_ROOT.rglob("*")
    if path.is_dir()
)

for folder in all_directories:
    relative_folder = folder.relative_to(DATA_ROOT)

    direct_files = [
        path for path in folder.iterdir()
        if path.is_file()
    ]

    if not direct_files:
        continue

    extension_counts = Counter(
        path.suffix.lower() if path.suffix else "<no extension>"
        for path in direct_files
    )

    print(f"\nFolder: {relative_folder}")
    print(f"Direct files: {len(direct_files):,}")

    for extension, count in sorted(extension_counts.items()):
        print(f"  {extension}: {count:,}")

    print("Examples:")
    for path in sorted(direct_files)[:3]:
        print(f"  - {path.name}")


Folder: fixations/test
Direct files: 5,000
  .mat: 5,000
Examples:
  - COCO_test2014_000000000001.mat
  - COCO_test2014_000000000063.mat
  - COCO_test2014_000000000188.mat

Folder: fixations/train
Direct files: 10,000
  .mat: 10,000
Examples:
  - COCO_train2014_000000000009.mat
  - COCO_train2014_000000000089.mat
  - COCO_train2014_000000000110.mat

Folder: fixations/val
Direct files: 5,000
  .mat: 5,000
Examples:
  - COCO_val2014_000000000133.mat
  - COCO_val2014_000000000164.mat
  - COCO_val2014_000000000192.mat

Folder: images/images/test
Direct files: 5,000
  .jpg: 5,000
Examples:
  - COCO_test2014_000000000001.jpg
  - COCO_test2014_000000000063.jpg
  - COCO_test2014_000000000188.jpg

Folder: images/images/train
Direct files: 10,000
  .jpg: 10,000
Examples:
  - COCO_train2014_000000000009.jpg
  - COCO_train2014_000000000089.jpg
  - COCO_train2014_000000000110.jpg

Folder: images/images/val
Direct files: 5,000
  .jpg: 5,000
Examples:
  - COCO_val2014_000000000133.jpg
  - COCO_val20

In [55]:
# ---------------------------------------------------------
# Infer split counts from folder and file names
# ---------------------------------------------------------

from collections import Counter

def infer_split(path):
    text = str(path.relative_to(DATA_ROOT)).lower()

    if "train" in text:
        return "train"

    if "val" in text or "validation" in text:
        return "val"

    if "test" in text:
        return "test"

    return "unknown"


jpg_split_counts = Counter(infer_split(path) for path in jpg_files)
png_split_counts = Counter(infer_split(path) for path in png_files)
mat_split_counts = Counter(infer_split(path) for path in mat_files)

print("JPG split counts:")
for split, count in sorted(jpg_split_counts.items()):
    print(f"  {split}: {count:,}")

print("\nPNG split counts:")
for split, count in sorted(png_split_counts.items()):
    print(f"  {split}: {count:,}")

print("\nMAT split counts:")
for split, count in sorted(mat_split_counts.items()):
    print(f"  {split}: {count:,}")

JPG split counts:
  test: 5,000
  train: 10,000
  val: 5,000

PNG split counts:
  train: 10,000
  val: 5,000

MAT split counts:
  test: 5,000
  train: 10,000
  val: 5,000


In [56]:
# ---------------------------------------------------------
# Verify that representative image and map files are valid
# ---------------------------------------------------------

from PIL import Image

sample_image_path = jpg_files[0]
sample_map_path = png_files[0]

with Image.open(sample_image_path) as image:
    image.load()

    print("Example RGB image")
    print(f"  Path: {sample_image_path.relative_to(DATA_ROOT)}")
    print(f"  Size: {image.size}")
    print(f"  Mode: {image.mode}")
    print(f"  Format: {image.format}")

with Image.open(sample_map_path) as saliency_map:
    saliency_map.load()

    print("\nExample saliency map")
    print(f"  Path: {sample_map_path.relative_to(DATA_ROOT)}")
    print(f"  Size: {saliency_map.size}")
    print(f"  Mode: {saliency_map.mode}")
    print(f"  Format: {saliency_map.format}")

Example RGB image
  Path: images/images/test/COCO_test2014_000000000001.jpg
  Size: (640, 480)
  Mode: RGB
  Format: JPEG

Example saliency map
  Path: maps/train/COCO_train2014_000000000009.png
  Size: (640, 480)
  Mode: L
  Format: PNG


In [57]:
# ---------------------------------------------------------
# Inspect filename conventions
# ---------------------------------------------------------

print("First 20 image filenames:")
for path in jpg_files[:20]:
    print(path.name)

print("\nFirst 20 map filenames:")
for path in png_files[:20]:
    print(path.name)

First 20 image filenames:
COCO_test2014_000000000001.jpg
COCO_test2014_000000000063.jpg
COCO_test2014_000000000188.jpg
COCO_test2014_000000000219.jpg
COCO_test2014_000000000318.jpg
COCO_test2014_000000000535.jpg
COCO_test2014_000000000549.jpg
COCO_test2014_000000000627.jpg
COCO_test2014_000000000694.jpg
COCO_test2014_000000001043.jpg
COCO_test2014_000000001257.jpg
COCO_test2014_000000001266.jpg
COCO_test2014_000000001410.jpg
COCO_test2014_000000001414.jpg
COCO_test2014_000000001620.jpg
COCO_test2014_000000001624.jpg
COCO_test2014_000000001671.jpg
COCO_test2014_000000002063.jpg
COCO_test2014_000000002166.jpg
COCO_test2014_000000002310.jpg

First 20 map filenames:
COCO_train2014_000000000009.png
COCO_train2014_000000000089.png
COCO_train2014_000000000110.png
COCO_train2014_000000000307.png
COCO_train2014_000000000321.png
COCO_train2014_000000000332.png
COCO_train2014_000000000349.png
COCO_train2014_000000000382.png
COCO_train2014_000000000389.png
COCO_train2014_000000000419.png
COCO_trai

In [58]:
# ---------------------------------------------------------
# Collect files and infer official SALICON split
# ---------------------------------------------------------

jpg_files = sorted(DATA_ROOT.rglob("*.jpg"))
png_files = sorted(DATA_ROOT.rglob("*.png"))
mat_files = sorted(DATA_ROOT.rglob("*.mat"))

def infer_official_split(path):
    text = str(path.relative_to(DATA_ROOT)).lower()

    if "train" in text:
        return "train"

    if "val" in text or "validation" in text:
        return "val"

    if "test" in text:
        return "test"

    return "unknown"


def count_by_split(paths):
    return Counter(infer_official_split(path) for path in paths)


print("RGB images:")
for split, count in sorted(count_by_split(jpg_files).items()):
    print(f"  {split}: {count:,}")

print("\nSaliency maps:")
for split, count in sorted(count_by_split(png_files).items()):
    print(f"  {split}: {count:,}")

print("\nFixation files:")
for split, count in sorted(count_by_split(mat_files).items()):
    print(f"  {split}: {count:,}")

RGB images:
  test: 5,000
  train: 10,000
  val: 5,000

Saliency maps:
  train: 10,000
  val: 5,000

Fixation files:
  test: 5,000
  train: 10,000
  val: 5,000


In [59]:
# ---------------------------------------------------------
# Inspect image and map filename conventions
# ---------------------------------------------------------

train_images = [
    path for path in jpg_files
    if infer_official_split(path) == "train"
]

val_images = [
    path for path in jpg_files
    if infer_official_split(path) == "val"
]

train_maps = [
    path for path in png_files
    if infer_official_split(path) == "train"
]

val_maps = [
    path for path in png_files
    if infer_official_split(path) == "val"
]

print("Training images:")
for path in train_images[:5]:
    print(" ", path.relative_to(DATA_ROOT))

print("\nTraining maps:")
for path in train_maps[:5]:
    print(" ", path.relative_to(DATA_ROOT))

print("\nValidation images:")
for path in val_images[:5]:
    print(" ", path.relative_to(DATA_ROOT))

print("\nValidation maps:")
for path in val_maps[:5]:
    print(" ", path.relative_to(DATA_ROOT))

Training images:
  images/images/train/COCO_train2014_000000000009.jpg
  images/images/train/COCO_train2014_000000000089.jpg
  images/images/train/COCO_train2014_000000000110.jpg
  images/images/train/COCO_train2014_000000000307.jpg
  images/images/train/COCO_train2014_000000000321.jpg

Training maps:
  maps/train/COCO_train2014_000000000009.png
  maps/train/COCO_train2014_000000000089.png
  maps/train/COCO_train2014_000000000110.png
  maps/train/COCO_train2014_000000000307.png
  maps/train/COCO_train2014_000000000321.png

Validation images:
  images/images/val/COCO_val2014_000000000133.jpg
  images/images/val/COCO_val2014_000000000164.jpg
  images/images/val/COCO_val2014_000000000192.jpg
  images/images/val/COCO_val2014_000000000196.jpg
  images/images/val/COCO_val2014_000000000208.jpg

Validation maps:
  maps/val/COCO_val2014_000000000133.png
  maps/val/COCO_val2014_000000000164.png
  maps/val/COCO_val2014_000000000192.png
  maps/val/COCO_val2014_000000000196.png
  maps/val/COCO_val2

In [60]:
# ---------------------------------------------------------
# Pair RGB images and saliency maps using COCO numeric IDs
# ---------------------------------------------------------

import re
from collections import defaultdict

def extract_sample_id(path):
    digit_groups = re.findall(r"\d+", path.stem)

    if not digit_groups:
        raise ValueError(
            f"No numeric sample ID found in filename: {path.name}"
        )

    # COCO sample ID is normally the final numeric group.
    return digit_groups[-1].lstrip("0") or "0"


def index_by_id(paths):
    index = {}
    duplicates = defaultdict(list)

    for path in paths:
        sample_id = extract_sample_id(path)

        if sample_id in index:
            duplicates[sample_id].append(path)
        else:
            index[sample_id] = path

    return index, duplicates


train_image_index, train_image_duplicates = index_by_id(train_images)
train_map_index, train_map_duplicates = index_by_id(train_maps)

val_image_index, val_image_duplicates = index_by_id(val_images)
val_map_index, val_map_duplicates = index_by_id(val_maps)

In [61]:
# ---------------------------------------------------------
# Validate image-map correspondence
# ---------------------------------------------------------

def audit_pairing(image_index, map_index, image_duplicates, map_duplicates, split):
    image_ids = set(image_index)
    map_ids = set(map_index)

    valid_ids = sorted(image_ids & map_ids)
    images_without_maps = sorted(image_ids - map_ids)
    maps_without_images = sorted(map_ids - image_ids)

    print(f"\n{split.upper()} PAIRING AUDIT")
    print(f"Images: {len(image_ids):,}")
    print(f"Maps: {len(map_ids):,}")
    print(f"Valid pairs: {len(valid_ids):,}")
    print(f"Images without maps: {len(images_without_maps):,}")
    print(f"Maps without images: {len(maps_without_images):,}")
    print(f"Duplicate image IDs: {len(image_duplicates):,}")
    print(f"Duplicate map IDs: {len(map_duplicates):,}")

    if images_without_maps[:5]:
        print("Example images without maps:", images_without_maps[:5])

    if maps_without_images[:5]:
        print("Example maps without images:", maps_without_images[:5])

    return valid_ids


valid_train_ids = audit_pairing(
    train_image_index,
    train_map_index,
    train_image_duplicates,
    train_map_duplicates,
    "train",
)

valid_official_val_ids = audit_pairing(
    val_image_index,
    val_map_index,
    val_image_duplicates,
    val_map_duplicates,
    "official validation",
)


TRAIN PAIRING AUDIT
Images: 10,000
Maps: 10,000
Valid pairs: 10,000
Images without maps: 0
Maps without images: 0
Duplicate image IDs: 0
Duplicate map IDs: 0

OFFICIAL VALIDATION PAIRING AUDIT
Images: 5,000
Maps: 5,000
Valid pairs: 5,000
Images without maps: 0
Maps without images: 0
Duplicate image IDs: 0
Duplicate map IDs: 0


In [62]:
# ---------------------------------------------------------
# Create verified paired records
# ---------------------------------------------------------

train_pairs = [
    {
        "sample_id": sample_id,
        "image_path": train_image_index[sample_id],
        "map_path": train_map_index[sample_id],
        "official_split": "train",
    }
    for sample_id in valid_train_ids
]

official_val_pairs = [
    {
        "sample_id": sample_id,
        "image_path": val_image_index[sample_id],
        "map_path": val_map_index[sample_id],
        "official_split": "val",
    }
    for sample_id in valid_official_val_ids
]

print(f"Verified training pairs: {len(train_pairs):,}")
print(f"Verified official validation pairs: {len(official_val_pairs):,}")

Verified training pairs: 10,000
Verified official validation pairs: 5,000


In [63]:
# ---------------------------------------------------------
# Final gate before saving the dataset manifests and splits
# ---------------------------------------------------------

required_variables = [
    "train_pairs",
    "official_val_pairs",
    "train_image_duplicates",
    "train_map_duplicates",
    "val_image_duplicates",
    "val_map_duplicates",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "The pairing-audit cells must be run first. "
        f"Missing variables: {missing_variables}"
    )

train_ids = [record["sample_id"] for record in train_pairs]
official_val_ids = [
    record["sample_id"]
    for record in official_val_pairs
]

assert len(train_ids) == len(set(train_ids)), \
    "Duplicate sample IDs exist in the training records."

assert len(official_val_ids) == len(set(official_val_ids)), \
    "Duplicate sample IDs exist in the official-validation records."

assert not train_image_duplicates, \
    "Duplicate training-image IDs were detected."

assert not train_map_duplicates, \
    "Duplicate training-map IDs were detected."

assert not val_image_duplicates, \
    "Duplicate validation-image IDs were detected."

assert not val_map_duplicates, \
    "Duplicate validation-map IDs were detected."

assert len(train_pairs) == 10_000, (
    f"Expected 10,000 training pairs, found {len(train_pairs):,}."
)

assert len(official_val_pairs) == 5_000, (
    "Expected 5,000 official-validation pairs, "
    f"found {len(official_val_pairs):,}."
)

for record in train_pairs + official_val_pairs:
    if not record["image_path"].is_file():
        raise FileNotFoundError(record["image_path"])

    if not record["map_path"].is_file():
        raise FileNotFoundError(record["map_path"])

print("Pairing gate passed.")
print(f"Verified training pairs: {len(train_pairs):,}")
print(
    "Verified official-validation pairs: "
    f"{len(official_val_pairs):,}"
)

Pairing gate passed.
Verified training pairs: 10,000
Verified official-validation pairs: 5,000


In [64]:
# ---------------------------------------------------------
# Create a deterministic image-map contact sheet
# ---------------------------------------------------------

import random
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
CONTACT_SHEET_PATH = FIGURE_ROOT / "data_contact_sheet.png"
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

rng = random.Random(SEED)

selected_train = rng.sample(train_pairs, 10)
selected_official_val = rng.sample(official_val_pairs, 10)

selected_records = [
    {**record, "display_split": "official_train"}
    for record in selected_train
] + [
    {**record, "display_split": "official_val"}
    for record in selected_official_val
]

fig, axes = plt.subplots(
    nrows=len(selected_records),
    ncols=3,
    figsize=(12, 4 * len(selected_records)),
)

for row, record in enumerate(selected_records):
    with Image.open(record["image_path"]) as image_file:
        rgb = image_file.convert("RGB")
        rgb_array = np.asarray(rgb)

    with Image.open(record["map_path"]) as map_file:
        saliency = map_file.convert("L")

        # Resize only for this visualization.
        # The original file is not changed.
        saliency = saliency.resize(
            rgb.size,
            resample=Image.Resampling.BILINEAR,
        )
        saliency_array = np.asarray(
            saliency,
            dtype=np.float32,
        )

    saliency_max = float(saliency_array.max())

    if saliency_max > 0:
        saliency_display = saliency_array / saliency_max
    else:
        saliency_display = saliency_array

    axes[row, 0].imshow(rgb_array)
    axes[row, 0].set_title(
        f"{record['display_split']} | ID {record['sample_id']}\nRGB"
    )
    axes[row, 0].axis("off")

    axes[row, 1].imshow(saliency_display, cmap="gray")
    axes[row, 1].set_title("Ground-truth saliency map")
    axes[row, 1].axis("off")

    axes[row, 2].imshow(rgb_array)
    axes[row, 2].imshow(
        saliency_display,
        cmap="jet",
        alpha=0.45,
    )
    axes[row, 2].set_title("Alignment overlay")
    axes[row, 2].axis("off")

plt.tight_layout()
plt.savefig(
    CONTACT_SHEET_PATH,
    dpi=150,
    bbox_inches="tight",
)
plt.show()
plt.close(fig)

print(f"Contact sheet saved to:\n{CONTACT_SHEET_PATH}")

Output hidden; open in https://colab.research.google.com to view.

In [65]:
# ---------------------------------------------------------
# Save the complete labelled-dataset manifest
# ---------------------------------------------------------

import csv

SPLIT_ROOT.mkdir(parents=True, exist_ok=True)

DATASET_MANIFEST_PATH = SPLIT_ROOT / "dataset_manifest.csv"

manifest_fields = [
    "sample_id",
    "official_split",
    "image_relpath",
    "map_relpath",
]

all_manifest_rows = []

for record in train_pairs:
    all_manifest_rows.append({
        "sample_id": record["sample_id"],
        "official_split": "train",
        "image_relpath": record["image_path"]
            .relative_to(DATA_ROOT)
            .as_posix(),
        "map_relpath": record["map_path"]
            .relative_to(DATA_ROOT)
            .as_posix(),
    })

for record in official_val_pairs:
    all_manifest_rows.append({
        "sample_id": record["sample_id"],
        "official_split": "val",
        "image_relpath": record["image_path"]
            .relative_to(DATA_ROOT)
            .as_posix(),
        "map_relpath": record["map_path"]
            .relative_to(DATA_ROOT)
            .as_posix(),
    })

with DATASET_MANIFEST_PATH.open(
    "w",
    newline="",
    encoding="utf-8",
) as file:
    writer = csv.DictWriter(
        file,
        fieldnames=manifest_fields,
    )
    writer.writeheader()
    writer.writerows(all_manifest_rows)

print(f"Dataset manifest saved to:\n{DATASET_MANIFEST_PATH}")
print(f"Rows saved: {len(all_manifest_rows):,}")

Dataset manifest saved to:
/content/drive/MyDrive/saliency_project/splits/dataset_manifest.csv
Rows saved: 15,000


In [66]:
# ---------------------------------------------------------
# Create or load the deterministic project splits
# ---------------------------------------------------------

import random

TRAIN_SPLIT_PATH = SPLIT_ROOT / "train.txt"
VAL_SPLIT_PATH = SPLIT_ROOT / "val.txt"
TEST_SPLIT_PATH = SPLIT_ROOT / "test.txt"

split_paths = {
    "train": TRAIN_SPLIT_PATH,
    "val": VAL_SPLIT_PATH,
    "test": TEST_SPLIT_PATH,
}

existing_status = {
    name: path.exists()
    for name, path in split_paths.items()
}


def write_ids(path, ids):
    with path.open("w", encoding="utf-8") as file:
        for sample_id in ids:
            file.write(f"{sample_id}\n")


def read_ids(path):
    with path.open("r", encoding="utf-8") as file:
        return [
            line.strip()
            for line in file
            if line.strip()
        ]


if all(existing_status.values()):
    print("Existing split files found. Loading without modification.")

    project_train_ids = read_ids(TRAIN_SPLIT_PATH)
    project_val_ids = read_ids(VAL_SPLIT_PATH)
    project_test_ids = read_ids(TEST_SPLIT_PATH)

elif any(existing_status.values()):
    raise RuntimeError(
        "Only some split files exist. "
        "Do not create or overwrite a partial split.\n"
        f"Status: {existing_status}"
    )

else:
    project_train_ids = sorted(train_ids)

    shuffled_official_val_ids = sorted(official_val_ids)

    split_rng = random.Random(SEED)
    split_rng.shuffle(shuffled_official_val_ids)

    project_val_ids = sorted(
        shuffled_official_val_ids[:2500]
    )

    project_test_ids = sorted(
        shuffled_official_val_ids[2500:]
    )

    write_ids(TRAIN_SPLIT_PATH, project_train_ids)
    write_ids(VAL_SPLIT_PATH, project_val_ids)
    write_ids(TEST_SPLIT_PATH, project_test_ids)

    print("New deterministic split files created.")

print(f"Project train: {len(project_train_ids):,}")
print(f"Project validation: {len(project_val_ids):,}")
print(f"Project internal test: {len(project_test_ids):,}")

Existing split files found. Loading without modification.
Project train: 10,000
Project validation: 2,500
Project internal test: 2,500


In [67]:
# ---------------------------------------------------------
# Validate split integrity
# ---------------------------------------------------------

train_id_set = set(project_train_ids)
val_id_set = set(project_val_ids)
test_id_set = set(project_test_ids)

official_train_id_set = set(train_ids)
official_val_id_set = set(official_val_ids)

assert len(project_train_ids) == 10_000
assert len(project_val_ids) == 2_500
assert len(project_test_ids) == 2_500

assert len(train_id_set) == len(project_train_ids), \
    "Duplicate IDs exist in train.txt."

assert len(val_id_set) == len(project_val_ids), \
    "Duplicate IDs exist in val.txt."

assert len(test_id_set) == len(project_test_ids), \
    "Duplicate IDs exist in test.txt."

assert train_id_set == official_train_id_set, (
    "train.txt does not exactly match the verified "
    "official-training IDs."
)

assert val_id_set.isdisjoint(test_id_set), (
    "Internal validation and test sets overlap."
)

assert val_id_set | test_id_set == official_val_id_set, (
    "Validation and test do not exactly cover the verified "
    "official-validation IDs."
)

print("Split integrity checks passed.")
print("Internal validation/test overlap: 0")
print(
    "Official-validation IDs represented: "
    f"{len(val_id_set | test_id_set):,}"
)

Split integrity checks passed.
Internal validation/test overlap: 0
Official-validation IDs represented: 5,000


In [68]:
# ---------------------------------------------------------
# Save train, validation, and test manifests
# ---------------------------------------------------------

train_record_by_id = {
    record["sample_id"]: record
    for record in train_pairs
}

official_val_record_by_id = {
    record["sample_id"]: record
    for record in official_val_pairs
}

split_manifest_fields = [
    "sample_id",
    "project_split",
    "official_split",
    "image_relpath",
    "map_relpath",
]


def build_split_rows(
    ids,
    record_index,
    project_split,
    official_split,
):
    rows = []

    for sample_id in ids:
        record = record_index[sample_id]

        rows.append({
            "sample_id": sample_id,
            "project_split": project_split,
            "official_split": official_split,
            "image_relpath": record["image_path"]
                .relative_to(DATA_ROOT)
                .as_posix(),
            "map_relpath": record["map_path"]
                .relative_to(DATA_ROOT)
                .as_posix(),
        })

    return rows


project_train_rows = build_split_rows(
    project_train_ids,
    train_record_by_id,
    project_split="train",
    official_split="train",
)

project_val_rows = build_split_rows(
    project_val_ids,
    official_val_record_by_id,
    project_split="val",
    official_split="val",
)

project_test_rows = build_split_rows(
    project_test_ids,
    official_val_record_by_id,
    project_split="test",
    official_split="val",
)

manifest_outputs = {
    SPLIT_ROOT / "train_manifest.csv": project_train_rows,
    SPLIT_ROOT / "val_manifest.csv": project_val_rows,
    SPLIT_ROOT / "test_manifest.csv": project_test_rows,
}

for output_path, rows in manifest_outputs.items():
    with output_path.open(
        "w",
        newline="",
        encoding="utf-8",
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=split_manifest_fields,
        )
        writer.writeheader()
        writer.writerows(rows)

    print(f"Saved {len(rows):,} rows -> {output_path.name}")

Saved 10,000 rows -> train_manifest.csv
Saved 2,500 rows -> val_manifest.csv
Saved 2,500 rows -> test_manifest.csv


In [69]:
# ---------------------------------------------------------
# Save reproducibility metadata and file hashes
# ---------------------------------------------------------

import hashlib
import json
from datetime import datetime, timezone


def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for block in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


split_metadata = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "seed": SEED,
    "protocol": {
        "training": (
            "All verified official SALICON training pairs"
        ),
        "validation": (
            "First 2500 IDs after deterministic shuffle "
            "of verified official-validation IDs"
        ),
        "test": (
            "Remaining 2500 IDs after deterministic shuffle "
            "of verified official-validation IDs"
        ),
        "official_test_used": False,
    },
    "counts": {
        "train": len(project_train_ids),
        "val": len(project_val_ids),
        "test": len(project_test_ids),
    },
    "sha256": {
        "train.txt": sha256_file(TRAIN_SPLIT_PATH),
        "val.txt": sha256_file(VAL_SPLIT_PATH),
        "test.txt": sha256_file(TEST_SPLIT_PATH),
        "dataset_manifest.csv": sha256_file(
            DATASET_MANIFEST_PATH
        ),
        "train_manifest.csv": sha256_file(
            SPLIT_ROOT / "train_manifest.csv"
        ),
        "val_manifest.csv": sha256_file(
            SPLIT_ROOT / "val_manifest.csv"
        ),
        "test_manifest.csv": sha256_file(
            SPLIT_ROOT / "test_manifest.csv"
        ),
    },
}

SPLIT_METADATA_PATH = SPLIT_ROOT / "split_metadata.json"

with SPLIT_METADATA_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        split_metadata,
        file,
        indent=2,
    )

print(f"Split metadata saved to:\n{SPLIT_METADATA_PATH}")

Split metadata saved to:
/content/drive/MyDrive/saliency_project/splits/split_metadata.json


In [70]:
# ---------------------------------------------------------
# Save machine-readable dataset audit
# ---------------------------------------------------------

LOG_ROOT.mkdir(parents=True, exist_ok=True)

DATA_AUDIT_PATH = LOG_ROOT / "data_audit.json"

dataset_audit = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "dataset_root": str(DATA_ROOT),
    "raw_download": {
        "total_files": 55_000,
        "total_size_gb": 3.97,
        "jpg_files": 20_000,
        "png_files": 15_000,
        "mat_files": 20_000,
    },
    "verified_labelled_pairs": {
        "official_train": len(train_pairs),
        "official_val": len(official_val_pairs),
        "total": len(train_pairs) + len(official_val_pairs),
    },
    "pairing_errors": {
        "duplicate_train_image_ids": len(
            train_image_duplicates
        ),
        "duplicate_train_map_ids": len(
            train_map_duplicates
        ),
        "duplicate_val_image_ids": len(
            val_image_duplicates
        ),
        "duplicate_val_map_ids": len(
            val_map_duplicates
        ),
    },
    "project_split": {
        "seed": SEED,
        "train": len(project_train_ids),
        "val": len(project_val_ids),
        "test": len(project_test_ids),
        "official_test_used": False,
    },
    "saved_outputs": {
        "contact_sheet": str(CONTACT_SHEET_PATH),
        "dataset_manifest": str(DATASET_MANIFEST_PATH),
        "split_metadata": str(SPLIT_METADATA_PATH),
    },
}

with DATA_AUDIT_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        dataset_audit,
        file,
        indent=2,
    )

print(f"Dataset audit saved to:\n{DATA_AUDIT_PATH}")

Dataset audit saved to:
/content/drive/MyDrive/saliency_project/logs/data_audit.json


In [71]:
# ---------------------------------------------------------
# Final data-stage readiness check
# ---------------------------------------------------------

def verify_manifest(manifest_path, expected_rows):
    with manifest_path.open(
        "r",
        encoding="utf-8",
    ) as file:
        rows = list(csv.DictReader(file))

    assert len(rows) == expected_rows, (
        f"{manifest_path.name}: expected {expected_rows:,} rows, "
        f"found {len(rows):,}."
    )

    for row in rows:
        image_path = DATA_ROOT / row["image_relpath"]
        map_path = DATA_ROOT / row["map_relpath"]

        if not image_path.is_file():
            raise FileNotFoundError(image_path)

        if not map_path.is_file():
            raise FileNotFoundError(map_path)

    print(
        f"{manifest_path.name}: "
        f"{len(rows):,} valid records"
    )


verify_manifest(
    SPLIT_ROOT / "train_manifest.csv",
    10_000,
)

verify_manifest(
    SPLIT_ROOT / "val_manifest.csv",
    2_500,
)

verify_manifest(
    SPLIT_ROOT / "test_manifest.csv",
    2_500,
)

required_outputs = [
    CONTACT_SHEET_PATH,
    DATA_AUDIT_PATH,
    DATASET_MANIFEST_PATH,
    TRAIN_SPLIT_PATH,
    VAL_SPLIT_PATH,
    TEST_SPLIT_PATH,
    SPLIT_ROOT / "train_manifest.csv",
    SPLIT_ROOT / "val_manifest.csv",
    SPLIT_ROOT / "test_manifest.csv",
    SPLIT_METADATA_PATH,
]

missing_outputs = [
    path
    for path in required_outputs
    if not path.exists()
]

assert not missing_outputs, (
    f"Missing outputs: {missing_outputs}"
)

print("\nDATA PREPARATION COMPLETE")
print("The project is ready for Dataset/DataLoader implementation.")

train_manifest.csv: 10,000 valid records
val_manifest.csv: 2,500 valid records
test_manifest.csv: 2,500 valid records

DATA PREPARATION COMPLETE
The project is ready for Dataset/DataLoader implementation.


In [72]:
# ---------------------------------------------------------
# Create a visible test file directly in Google Drive
# ---------------------------------------------------------

SYNC_TEST_PATH = DRIVE_ROOT / "drive_sync_test.txt"

SYNC_TEST_PATH.write_text(
    "This file was created from Google Colab.\n",
    encoding="utf-8",
)

# Ask the operating system to flush pending writes.
os.sync()

print(f"Created: {SYNC_TEST_PATH}")
print(f"Exists from Colab: {SYNC_TEST_PATH.exists()}")
print(f"Size: {SYNC_TEST_PATH.stat().st_size} bytes")

Created: /content/drive/MyDrive/saliency_project/drive_sync_test.txt
Exists from Colab: True
Size: 41 bytes


In [73]:
# ---------------------------------------------------------
# Verify whether saved files are truly inside Google Drive
# ---------------------------------------------------------

from pathlib import Path
import os

paths_to_check = {
    "project root": DRIVE_ROOT,
    "splits": SPLIT_ROOT,
    "logs": LOG_ROOT,
    "figures": FIGURE_ROOT,
}

for name, path in paths_to_check.items():
    path = Path(path)

    print(f"\n{name.upper()}")
    print(f"Path: {path}")
    print(f"Resolved path: {path.resolve()}")
    print(f"Exists: {path.exists()}")
    print(
        "Inside mounted Drive:",
        str(path.resolve()).startswith("/content/drive/MyDrive/")
    )

    if path.exists():
        files = sorted(
            item for item in path.rglob("*")
            if item.is_file()
        )

        print(f"Files found: {len(files)}")

        for item in files[:15]:
            print(f"  - {item.name}")


PROJECT ROOT
Path: /content/drive/MyDrive/saliency_project
Resolved path: /content/drive/MyDrive/saliency_project
Exists: True
Inside mounted Drive: True
Files found: 55015
  - DRIVE_TEST_20260712_143643.txt
  - FIND_ME_FROM_COLAB_20260712_141539.txt
  - COCO_test2014_000000000001.mat
  - COCO_test2014_000000000063.mat
  - COCO_test2014_000000000188.mat
  - COCO_test2014_000000000219.mat
  - COCO_test2014_000000000318.mat
  - COCO_test2014_000000000535.mat
  - COCO_test2014_000000000549.mat
  - COCO_test2014_000000000627.mat
  - COCO_test2014_000000000694.mat
  - COCO_test2014_000000001043.mat
  - COCO_test2014_000000001257.mat
  - COCO_test2014_000000001266.mat
  - COCO_test2014_000000001410.mat

SPLITS
Path: /content/drive/MyDrive/saliency_project/splits
Resolved path: /content/drive/MyDrive/saliency_project/splits
Exists: True
Inside mounted Drive: True
Files found: 8
  - dataset_manifest.csv
  - split_metadata.json
  - test.txt
  - test_manifest.csv
  - train.txt
  - train_manifes

In [74]:
# ---------------------------------------------------------
# Final persistent-storage verification
# ---------------------------------------------------------

import os

os.sync()

expected_files = [
    SPLIT_ROOT / "dataset_manifest.csv",
    SPLIT_ROOT / "train.txt",
    SPLIT_ROOT / "val.txt",
    SPLIT_ROOT / "test.txt",
    SPLIT_ROOT / "train_manifest.csv",
    SPLIT_ROOT / "val_manifest.csv",
    SPLIT_ROOT / "test_manifest.csv",
    SPLIT_ROOT / "split_metadata.json",
    LOG_ROOT / "data_audit.json",
    FIGURE_ROOT / "data_contact_sheet.png",
]

print("Persistent Google Drive verification")
print("------------------------------------")

for path in expected_files:
    inside_drive = str(path).startswith("/content/drive/MyDrive/")
    status = "OK" if path.is_file() and inside_drive else "MISSING"

    print(f"{status:8} {path.relative_to(DRIVE_ROOT)}")

missing = [
    path for path in expected_files
    if not path.is_file()
]

assert not missing, f"Missing persistent files: {missing}"

print("\nAll data-preparation outputs are stored in mounted Google Drive.")

Persistent Google Drive verification
------------------------------------
OK       splits/dataset_manifest.csv
OK       splits/train.txt
OK       splits/val.txt
OK       splits/test.txt
OK       splits/train_manifest.csv
OK       splits/val_manifest.csv
OK       splits/test_manifest.csv
OK       splits/split_metadata.json
OK       logs/data_audit.json
OK       outputs/figures/data_contact_sheet.png

All data-preparation outputs are stored in mounted Google Drive.
